In [1]:
import numpy as np
import pandas as pd
import patsy
from tensorzinb.tensorzinb import TensorZINB


2025-04-28 18:12:40.229737: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
fake_cres=pd.read_csv("fake_cres.csv").drop("Unnamed: 0",axis=1)
fake_cres

,CRE,Cell_type,replicate_ID,umi_count
0,nobody,brain,1,0
1,nobody,brain,1,0
2,nobody,brain,1,0
3,nobody,brain,1,0
4,nobody,brain,1,0
...,...,...,...,...
14307,neurogene,blood,3,7
14308,neurogene,blood,3,26
14309,neurogene,blood,3,7
14310,neurogene,blood,3,15


In [3]:
fake_cres_munged=fake_cres
fake_cres_munged["replicate_ID"]=fake_cres_munged["replicate_ID"].map({1:"rep1",2:"rep2",3:"rep3"})

In [4]:
from formulaic import Formula

In [5]:
fake_cres_munged

,CRE,Cell_type,replicate_ID,umi_count
0,nobody,brain,rep1,0
1,nobody,brain,rep1,0
2,nobody,brain,rep1,0
3,nobody,brain,rep1,0
4,nobody,brain,rep1,0
...,...,...,...,...
14307,neurogene,blood,rep3,7
14308,neurogene,blood,rep3,26
14309,neurogene,blood,rep3,7
14310,neurogene,blood,rep3,15


In [6]:
#dense dmatrix approach : works fine for small datasets, but for very large...

#y, X = patsy.dmatrices("umi_count ~ C(CRE)*C(Cell_type)-1",
#                        fake_cres_munged, return_type='dataframe')
#Z = patsy.dmatrix("C(replicate_ID)", fake_cres_munged, return_type='dataframe')

#zinbo=TensorZINB(y["umi_count"].to_numpy().reshape((-1,1)),X,exog_infl=Z.to_numpy())#,same_dispersion=True
#zinb_result=zinbo.fit(init_method="nb")

### simple `formulaic` approach, still using pandas

y, X=Formula("umi_count ~ C(CRE)*C(Cell_type) - 1").get_model_matrix(fake_cres_munged,output='pandas')
Z=Formula('C(replicate_ID)').get_model_matrix(fake_cres_munged,output='pandas')


zinbo=TensorZINB(y["umi_count"].to_numpy().reshape((-1,1)),X.to_numpy(),exog_infl=Z.to_numpy())#,same_dispersion=True
zinb_result=zinbo.fit(init_method="nb")


2025-04-28 14:52:33.795395: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-04-28 14:52:33.795435: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (r814u03n08.mccleary.ycrc.yale.edu): /proc/driver/nvidia/version does not exist
2025-04-28 14:52:34.044553: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-28 14:52:34.824332: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:354] MLIR V1 optimization pass is not enabled


In [8]:
dir(zinbo)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_compute_pi_init',
 '_estimate_dispersion',
 '_nb_init',
 '_no_exog_c',
 '_no_exog_infl',
 '_no_exog_infl_c',
 '_poisson_init',
 '_poisson_init_each',
 'df_model',
 'df_model_each',
 'endog',
 'exog',
 'exog_c',
 'exog_infl',
 'exog_infl_c',
 'fit',
 'k_disperson',
 'k_exog',
 'k_exog_c',
 'k_exog_infl',
 'k_exog_infl_c',
 'loglike_method',
 'nb_only',
 'num_out',
 'num_sample',
 'same_dispersion']

In [32]:
zinb_result

{'llf_total': -21228.78548024269,
 'llfs': array([-21228.78548024]),
 'aic_total': 42485.57096048538,
 'aics': array([42485.57096049]),
 'df_model_total': 14,
 'df': 14,
 'weights': {'x_mu': array([[ 4.6391068 ],
         [ 2.739979  ],
         [ 0.665211  ],
         [ 4.6586823 ],
         [ 2.5245173 ],
         [ 0.03896479],
         [ 1.7861156 ],
         [-0.67995   ],
         [-1.2745893 ],
         [-0.26423696]], dtype=float32),
  'x_pi': array([[ 1.3560897],
         [-1.3262616],
         [ 0.8899107]], dtype=float32),
  'theta': array([[1.2496016]], dtype=float32)},
 'cpu_time': 1.8349146842956543,
 'num_sample': 14312,
 'epochs': 606}

# Recapitulating dispersion constant

In [33]:
np.exp(zinb_result['weights']['theta'][0])

array([3.4889526], dtype=float32)

If I am correct in exponentiating it (I think I am), that's pretty close to the ground truth value of 3.333

# Recapitulating zero-inflation parameters

In [34]:
#I think x_pi are the weights on 
zinb_result['weights']['x_pi']

array([[ 1.3560897],
       [-1.3262616],
       [ 0.8899107]], dtype=float32)

In [35]:
dropout_design_matrix=Z.drop_duplicates().to_numpy()

In [36]:
dropout_design_matrix

array([[1., 0., 0.],
       [1., 1., 0.],
       [1., 0., 1.]])

In [37]:
pis=dropout_design_matrix.dot(zinb_result['weights']['x_pi'])
pis

array([[1.35608971],
       [0.02982807],
       [2.24600041]])

In [38]:
#hrm. I bet these are bernouli constants passed through logit. 
#Let's undo w/ logistic function
1/(1+np.exp(-pis))

array([[0.79512344],
       [0.50745647],
       [0.90430498]])

If we assume the order is rep1, rep2, rep3, then the real values are 0.8, 0.5, 0.9.

That's pretty damn close!

# Recapitulating $\mu$s (mean parameters).

In [39]:
#we begin by getting all combinations of the predictors (which are of course all categorical) present in the data. 
minimal_nb_design = X.drop_duplicates()
minimal_nb_design

,C(CRE)[everybody],C(CRE)[neurogene],C(CRE)[nobody],C(CRE)[redgene],C(CRE)[somebody],C(Cell_type)[T.brain],C(CRE)[T.neurogene]:C(Cell_type)[T.brain],C(CRE)[T.nobody]:C(Cell_type)[T.brain],C(CRE)[T.redgene]:C(Cell_type)[T.brain],C(CRE)[T.somebody]:C(Cell_type)[T.brain]
0,0,0,1,0,0,1,0,1,0,0
440,0,0,0,0,1,1,0,0,0,1
904,1,0,0,0,0,1,0,0,0,0
1355,0,0,0,1,0,1,0,0,1,0
1798,0,1,0,0,0,1,1,0,0,0
2263,0,0,1,0,0,0,0,0,0,0
2762,0,0,0,0,1,0,0,0,0,0
3266,1,0,0,0,0,0,0,0,0,0
3767,0,0,0,1,0,0,0,0,0,0
4262,0,1,0,0,0,0,0,0,0,0


In [40]:
#examining the table above, we reconstruct the index
recapitulated_nb_rate=pd.DataFrame({"cre":["nobody","somebody","everybody","redgene","neurogene","nobody","somebody","everybody","redgene","neurogene"],
    "cell_type":["brain"]*5+["blood"]*5})
recapitulated_nb_rate

,cre,cell_type
0,nobody,brain
1,somebody,brain
2,everybody,brain
3,redgene,brain
4,neurogene,brain
5,nobody,blood
6,somebody,blood
7,everybody,blood
8,redgene,blood
9,neurogene,blood


In [41]:
zinb_result['weights']['x_mu']

array([[ 4.6391068 ],
       [ 2.739979  ],
       [ 0.665211  ],
       [ 4.6586823 ],
       [ 2.5245173 ],
       [ 0.03896479],
       [ 1.7861156 ],
       [-0.67995   ],
       [-1.2745893 ],
       [-0.26423696]], dtype=float32)

In [42]:
len(zinb_result['weights']['x_mu'])

10

That's the same as the number of columns in our design matrix. Assuming orientation was preserved...

In [43]:
np.exp(minimal_nb_design.dot(zinb_result['weights']['x_mu']))

,0
0,1.024522
440,9.966656
904,107.562443
1355,30.663034
1798,96.068306
2263,1.944901
2762,12.484867
3266,103.451898
3767,105.496982
4262,15.486660


In [44]:
recapitulated_nb_rate["expected_value"]=np.exp(minimal_nb_design.dot(zinb_result['weights']['x_mu'])).to_numpy()
#exponent to undo log link function
recapitulated_nb_rate

,cre,cell_type,expected_value
0,nobody,brain,1.024522
1,somebody,brain,9.966656
2,everybody,brain,107.562443
3,redgene,brain,30.663034
4,neurogene,brain,96.068306
5,nobody,blood,1.944901
6,somebody,blood,12.484867
7,everybody,blood,103.451898
8,redgene,blood,105.496982
9,neurogene,blood,15.486660


Comparing to the ground-truth:

In [45]:
REAL_data = {
    "CRE": ["nobody", "somebody", "everybody", "redgene", "neurogene", "nobody", "somebody", "everybody", "redgene", "neurogene"],
    "Cell-type": ["brain", "brain", "brain", "brain", "brain", "blood", "blood", "blood", "blood", "blood"],
    "mean": [1, 10, 114, 30, 99, 2, 12, 109, 112, 16]
}

# Creating the dataframe
pd.DataFrame(REAL_data)

,CRE,Cell-type,mean
0,nobody,brain,1
1,somebody,brain,10
2,everybody,brain,114
3,redgene,brain,30
4,neurogene,brain,99
5,nobody,blood,2
6,somebody,blood,12
7,everybody,blood,109
8,redgene,blood,112
9,neurogene,blood,16


Pretty good !